In [1]:
from khoi_dong import reset_nhan_vien, CHIEN_LUOC_BAN_DAU

for ten, (chuc_vu, chien_luoc) in CHIEN_LUOC_BAN_DAU.items():
    print(f"{ten} - {chuc_vu}")
    print(f"{chien_luoc[:150]}..")

print()
# reset_nhan_vien()

An - Nhân viên CSKH
Tôi là An, nhân viên CSKH lấy cảm hứng từ triết lý 'khách hàng là thượng đế'. Tôi ưu tiên xử lý đơn hàng nhanh chóng và chủ động liên hệ khách khi có ..
Binh - Chuyên viên Phân tích
Tôi là Bình, chuyên viên phân tích lấy cảm hứng từ tư duy dựa trên dữ liệu. Tôi đưa ra quyết định dựa trên số liệu: tỷ lệ chuyển đổi, doanh thu theo k..
Chi - Chuyên viên Vận hành
Tôi là Chi, chuyên viên vận hành theo hướng hệ thống và quy trình chuẩn. Tôi đảm bảo mọi đơn hàng được xử lý theo đúng quy trình, không có ngoại lệ. T..
Dung - Giám sát Hệ thống
Tôi là Dũng, giám sát hệ thống với tầm nhìn chiến lược dài hạn. Tôi theo dõi bức tranh toàn cảnh: xu hướng thị trường, vị thế cạnh tranh, và cơ hội tă..



In [3]:
# test custome tracer
from tracers_shop import tao_trace_id, LogTracer
from database_shop import ghi_log, doc_log

# Test tao trace_id
ten = "An"
tid = tao_trace_id(ten)
print(f"Trace ID cho {ten}:")
print(f"{tid}")

# Test extrace ten
after = tid.split("_", 1)[1]
ten_extract = after.split("0")[0]
print(f"Tên Extracted: '{ten_extract}' (phải là '{ten.lower()}')")

# Test ghi_log
ghi_log("An", "agent", "Test: An đang xử lý đơn hàng mới")
ghi_log("Binh", "tool", "Test: Bình gọi tool báo cáo doanh thu")
ghi_log("Chi", "trace", "Test: Chi hoàn thành rà soát quy trình")

# test doc_log
for row in doc_log("An", gioi_han=3):
    print(f" [{row['loai']}] {row['noi_dung']}")

Trace ID cho An:
trace_an0wyf34ggq5i3zpqsjgjms18j76lap0
Tên Extracted: 'an' (phải là 'an')
 [agent] Test: An đang xử lý đơn hàng mới


In [4]:
# Test voi mot nhan vien
from agents import add_trace_processor
from tracers_shop import LogTracer
from nhan_vien import NhanVien
from templates_shop import nhiem_vu_xu_ly_don

# Đăng ký custom tracer — chỉ cần gọi một lần cho toàn session
add_trace_processor(LogTracer())

print("Custom LogTracer đã được đăng ký.")
print("Mọi trace event từ agent SDK → sẽ được ghi vào san_van_cskh.db\n")

# Chạy nhân viên An với custom tracer active
an = NhanVien("An", chuc_vu="Nhân viên CSKH")
await an.run()

print("\n=== Log được ghi bởi tracer ===")
from database_shop import doc_log
for row in doc_log("An", gioi_han=10):
    print(f"  [{row['loai']:<8}] {row['noi_dung'][:80]}")

Custom LogTracer đã được đăng ký.
Mọi trace event từ agent SDK → sẽ được ghi vào san_van_cskh.db


[An] Nhiệm vụ: Xử lý đơn hàng

[An] Hoàn thành:
### Kết quả xử lý:

- **Số lượng đơn hàng đã xử lý**: 3 
- **Tổng giá trị đơn hàng**: 0 VNĐ (không có thông tin giá trị đơn hàng)
- **Tỷ giá USD hiện tại**: 
  - Mua TM: 26,107 VNĐ
  - Mua CK: 26,137 VNĐ
  - Bán: 26,357 VNĐ

Thông báo đã được gửi và báo cáo đã được lưu vào file `sandbox/bao_cao_xu_ly_an.md`....

=== Log được ghi bởi tracer ===
  [agent   ] Test: An đang xử lý đơn hàng mới


In [5]:
from mcp_params_shop import nhan_vien_mcp_server_params, nghien_cuu_mcp_server_params
from agents.mcp import MCPServerStdio

print("Đếm tổng số tools trong hệ thống...\n")

tat_ca_params = nhan_vien_mcp_server_params + nghien_cuu_mcp_server_params("test")
tong_tools = 0

for p in tat_ca_params:
    try:
        async with MCPServerStdio(params=p, client_session_timeout_seconds=20) as server:
            tools = await server.list_tools()
            ten_server = p["args"][-1].split("/")[-1].replace(".js","").replace(".py","")
            print(f"  {ten_server:<30} {len(tools):>3} tools")
            tong_tools += len(tools)
    except Exception as e:
        print(f"  [skip] {p['args'][-1].split('/')[-1]}: {e}")

print(f"\n{'─'*40}")
print(f"  Tổng: {len(tat_ca_params)} MCP servers, {tong_tools} tools")
print(f"\n4 nhân viên × {tong_tools} tools có thể sử dụng")
print("→ Đây là sức mạnh của multi-agent MCP system")

Đếm tổng số tools trong hệ thống...

  donhang_server                   5 tools
  dulieu_vn_server                 7 tools
  thong_bao_server                 2 tools
  sandbox                         14 tools
  index                            5 tools
  index                            9 tools

────────────────────────────────────────
  Tổng: 6 MCP servers, 42 tools

4 nhân viên × 42 tools có thể sử dụng
→ Đây là sức mạnh của multi-agent MCP system
